# 20 — 图基础模型与图—语言模型：迁移、提示与 Graph-RAG

这是课程主线的汇合课。我们先用统一结构特征和 GPS 编码器做跨图迁移，再实现可学习 graph prompt，最后构造一个可审计的 Graph-RAG 流水线。必须先立下边界：**一个 Graph Transformer 不是图基础模型，一次 source→target 微调也不是通用迁移证据。** 真正的 foundation model 需要大规模多域预训练、跨数据集/任务泛化、统一输入语义和系统评价。

## 1. 本课要区分的四件事

1. **Graph Transformer**：一种架构，通常结合局部消息传递、全局 attention 与位置/结构编码。
2. **图基础模型**：在多图、多域或多任务上预训练并能迁移的模型类别；规模不是唯一标准，但跨域证据不可缺少。
3. **图编码器 + LLM**：图编码器产生向量，经 projector 接入语言模型；需要模态对齐训练。
4. **图序列化 / Graph-RAG**：把检索到的节点、边和属性转成文本交给 LLM；实现简单但受上下文长度、顺序和信息损失限制。

知识图谱增强 LLM 既可能是训练时注入，也可能是推理时检索；两者不能混称。

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys, time
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.graph_classification import load_tu_dataset, stratified_graph_split
from crosscity.data.graph_foundation import (build_grounded_prompt, retrieve_graph_context,
    structuralize_graph)
from crosscity.models.graph_foundation import GraphTransferClassifier, UniversalGraphEncoder
from crosscity.training.graph_classification import train_graph_classifier
from crosscity.utils import seed_everything
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__)


## 2. 运行配置

Mac 模式减少 epoch、seed 和每类目标训练图数量；服务器正式模式使用完整配置。首次运行会下载 PROTEINS 与 MUTAG。GPS 的全局 attention 对单图节点数近似 $O(n^2)$，大图必须使用稀疏/线性 attention、分块或采样，不能直接照搬本课。

In [ ]:
RUN_FULL = False
SEEDS = [42, 43, 44] if RUN_FULL else [42]
SOURCE_EPOCHS = 200 if RUN_FULL else 20
TARGET_EPOCHS = 300 if RUN_FULL else 40
PATIENCE = 40 if RUN_FULL else 12
TARGET_PER_CLASS = 10 if RUN_FULL else 3
HIDDEN = 128 if RUN_FULL else 32
MAX_DEGREE = 8
STRUCTURAL_DIM = MAX_DEGREE + 3


## 3. 为什么需要统一输入语义？

不同图的原始特征可能分别表示原子类别、论文词袋或用户属性，维度相同也不代表语义相同。盲目复用第一层权重没有意义。本课把原属性替换为固定结构 schema：截断 degree one-hot、$\log(1+d_v)$ 和常数通道。这样 PROTEINS 与 MUTAG 可共享 encoder，但会丢失化学元素等重要信息。该取舍是教学用的可控接口，不是最佳分子表示。真正的通用模型还会使用 Laplacian/random-walk positional encoding、类型编码与领域 tokenizer。

In [ ]:
source_raw = load_tu_dataset(ROOT / 'data/raw/tu', 'PROTEINS')
target_raw = load_tu_dataset(ROOT / 'data/raw/tu', 'MUTAG')
source_split = stratified_graph_split(source_raw, seed=2026)
target_split = stratified_graph_split(target_raw, seed=2026)

def convert(subset):
    return [structuralize_graph(graph, MAX_DEGREE) for graph in subset]
source = {name: convert(getattr(source_split, name)) for name in ['train','validation','test']}
target = {name: convert(getattr(target_split, name)) for name in ['train','validation','test']}
pd.DataFrame([
    {'dataset':'PROTEINS/source','graphs':len(source_raw),'classes':source_raw.num_classes},
    {'dataset':'MUTAG/target','graphs':len(target_raw),'classes':target_raw.num_classes},
]).set_index('dataset')


## 4. GPS：局部消息 + 全局注意力

每个 GPS block 同时计算局部 GIN 消息和同一图内部的多头 self-attention，再通过残差与归一化组合。局部分支保留边归纳偏置，全局分支缩短远程通信路径，对第 18 课的过挤压有潜在帮助；但 attention 不自动理解结构，因此仍需要结构/位置编码。PyG 的 `batch` 阻止不同图之间互相注意。

## 5. 少样本目标划分

只缩小 MUTAG training 集，每类取固定数量；validation/test 不变。source 标签只能训练 source 任务，绝不能用于选择 target checkpoint。因为 MUTAG 很小，正式报告必须使用多 seed，并承认单一数据集对‘通用迁移’证据很弱。

In [ ]:
def few_graphs_per_class(graphs, count, seed):
    generator = torch.Generator().manual_seed(seed)
    selected = []
    labels = sorted({int(graph.y.item()) for graph in graphs})
    for label in labels:
        candidates = [graph for graph in graphs if int(graph.y.item()) == label]
        order = torch.randperm(len(candidates), generator=generator)[:count].tolist()
        selected.extend(candidates[index] for index in order)
    return selected

few_target_train = few_graphs_per_class(target['train'], TARGET_PER_CLASS, seed=2026)
print('target train/validation/test:', len(few_target_train), len(target['validation']), len(target['test']))
print('structural feature shape:', few_target_train[0].x.shape)


## 6. Smoke run

这一步只证明 batch、prompt、GPS 与训练器接口一致。可学习 prompt 是加到所有节点结构特征上的任务向量；它参数少，但表达能力有限。这里的 prompt tuning 还会训练新的分类头。

In [ ]:
smoke_loader = DataLoader(few_target_train, batch_size=4)
smoke_encoder = UniversalGraphEncoder(STRUCTURAL_DIM, 16, layers=1, heads=4, dropout=0)
smoke_model = GraphTransferClassifier(smoke_encoder, STRUCTURAL_DIM, 16,
    target_raw.num_classes, use_prompt=True)
smoke_model.freeze_encoder()
smoke_result = train_graph_classifier(smoke_model, smoke_loader, smoke_loader, smoke_loader,
    device=DEVICE, max_epochs=2, patience=2)
batch = next(iter(smoke_loader)).to(DEVICE)
print('graphs/logits:', batch.num_graphs, smoke_model(batch.x,batch.edge_index,batch.batch).shape)


## 7. Source 训练与 target 迁移

实验比较：`scratch` 从随机初始化学习；`transfer-linear` 冻结 source encoder 只训练 target head；`transfer-prompt` 冻结 encoder，训练 prompt+head；`transfer-finetune` 更新全部参数。架构、结构输入与 target 训练预算相同。这里 source 训练是有监督的，因此准确名称是**跨数据集迁移实验**，不是 foundation-model pretraining。

In [ ]:
records = []
for seed in SEEDS:
    seed_everything(seed)
    source_loaders = {
        'train': DataLoader(source['train'], batch_size=32, shuffle=True,
            generator=torch.Generator().manual_seed(seed)),
        'validation': DataLoader(source['validation'], batch_size=64),
        'test': DataLoader(source['test'], batch_size=64),
    }
    source_encoder = UniversalGraphEncoder(STRUCTURAL_DIM, HIDDEN, layers=3, heads=4)
    source_model = GraphTransferClassifier(source_encoder, STRUCTURAL_DIM, HIDDEN,
        source_raw.num_classes)
    train_graph_classifier(source_model, source_loaders['train'], source_loaders['validation'],
        source_loaders['test'], device=DEVICE, max_epochs=SOURCE_EPOCHS, patience=PATIENCE)
    source_state = deepcopy(source_model.encoder).cpu()

    target_train = DataLoader(few_target_train, batch_size=16, shuffle=True,
        generator=torch.Generator().manual_seed(seed))
    target_val = DataLoader(target['validation'], batch_size=64)
    target_test = DataLoader(target['test'], batch_size=64)
    variants = {}
    variants['scratch'] = GraphTransferClassifier(
        UniversalGraphEncoder(STRUCTURAL_DIM,HIDDEN,3,4), STRUCTURAL_DIM,HIDDEN,target_raw.num_classes)
    for name, prompt, freeze in [('transfer-linear',False,True),
                                  ('transfer-prompt',True,True),
                                  ('transfer-finetune',False,False)]:
        model = GraphTransferClassifier(deepcopy(source_state), STRUCTURAL_DIM, HIDDEN,
            target_raw.num_classes, use_prompt=prompt)
        if freeze: model.freeze_encoder()
        variants[name] = model
    for name, model in variants.items():
        seed_everything(seed)
        started = time.perf_counter()
        result = train_graph_classifier(model, target_train, target_val, target_test, device=DEVICE,
            max_epochs=TARGET_EPOCHS, patience=PATIENCE, learning_rate=0.001)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        records.append({'seed':seed,'method':name,'trainable_parameters':trainable,
            'validation_accuracy':result.validation_accuracy,'test_accuracy':result.test_accuracy,
            'seconds':time.perf_counter()-started})
transfer_results = pd.DataFrame(records)
transfer_results.groupby('method')[['validation_accuracy','trainable_parameters','seconds']].agg(['mean','std'])


## 8. 如何判定迁移是否成立

先在 validation 上冻结方案，再查看 test。`transfer-linear > scratch` 表示 source 表示可直接复用；只有 fine-tune 改善说明初始化有用但表示未对齐；prompt 改善且训练参数明显更少，说明参数高效适配值得继续研究。负迁移同样合理：两个数据集标签语义不同、结构统计不同，且 source 训练规模远不足以支撑‘基础模型’主张。

In [ ]:
transfer_results.groupby('method')[['validation_accuracy','test_accuracy',
    'trainable_parameters','seconds']].agg(['mean','std'])


## 9. 图与语言的三种接口

**编码器 + LLM**：$z_G=f_G(G)$，再用 projector 把图 token 映射到 LLM embedding；需要图文配对或指令数据对齐，单独拼接向量不会让 LLM 理解它。**序列化**：把节点、边、类型转为文本；无需训练 projector，但顺序敏感、token 成本高，可能丢失拓扑。**Graph-RAG**：先做语义检索，再沿图扩展邻域，只序列化与问题相关的子图；可给出证据，但检索召回决定回答上限。

## 10. 可审计的 Graph-RAG 最小实现

下面不调用外部 LLM。它把流程拆为：TF-IDF 找 seed → k-hop 扩展 → 确定性序列化 → grounded prompt。这样可以独立测量 seed recall、子图 recall、上下文长度和最终回答，而不是把所有错误归给 LLM。节点文本被视为不可信数据，用显式边界包裹，降低检索内容中的指令注入风险。

In [ ]:
node_texts = [
    'Alice is a researcher in graph neural networks.',
    'Bob studies protein structure.',
    'Carol collaborates on scalable graph learning.',
    'Paper-X presents neighbor sampling for large graphs.',
    'Lab-Y is located in Shanghai.',
]
edges = torch.tensor([[0,2,0,3,2,3,0,4],[2,0,3,0,3,2,4,0]])
question = 'Which work connected to Alice concerns scalable graph learning?'
context = retrieve_graph_context(question, node_texts, edges, top_k=2, hops=1)
prompt = build_grounded_prompt(question, context)
print('seed nodes:', context.seed_nodes, '| context nodes:', context.context_nodes)
print(prompt)


## 11. Graph-RAG 评价与安全

应分层报告：检索 seed 的 Recall@K、扩展后 gold evidence coverage、序列化 token/字符数、带引用的回答正确率，以及无答案问题上的拒答率。扩大 hops 会提高召回也会引入噪声和成本。图中可能包含私密字段、错误事实或恶意文本；部署前需要访问控制、来源与时间戳、字段白名单、上下文长度上限，以及把‘数据’与‘指令’分离。LLM 输出仍需链接回节点/边证据，不能把流畅文本当事实。

## 12. 什么证据才接近图基础模型？

至少需要：多域且规模充分的预训练图；统一但不抹掉语义的 tokenizer/schema；节点、边、图级和生成任务的迁移；unseen dataset 与分布外评价；scratch、单域预训练和同参数量基线；数据去重与污染审计；训练/推理成本；负迁移和失败域。模型变大、用了 Transformer、加入 prompt 或接上 LLM，都不是充分条件。

## 13. 课后练习与课程收束

1. 用 NCI1、MUTAG、PROTEINS 做 leave-one-dataset-out 迁移，不能只报告最有利方向。
2. 加入 Laplacian positional encoding，并审计特征向量符号/基底歧义。
3. 比较 prompt-only、head-only、full fine-tune 的参数量、显存和准确率。
4. 为 Graph-RAG 构造 gold evidence，画 hops 与 evidence recall/context size 的曲线。
5. 将同一子图用 edge list、邻接表和路径三种方式序列化，测试顺序敏感性。
6. 设计无答案与冲突事实案例，要求系统拒答或展示冲突来源。

至此课程从时间序列、静态/异构/动态图、采样与失效机制，走到自监督、跨图迁移和图—语言接口。最终能力不是记住某个模型名字，而是能判断：图怎样定义、信息怎样传播、评价是否泄漏，以及复杂系统是否真的提供可复现的增益。